### **1. Load and inspect the file**


In [6]:
import pandas as pd
import numpy as np
import nbformat 

labs = pd.read_csv("/Users/are/Development/DA229/PythonHackathon/Python_Hackathon_Sep_2026/cardiac_failure/labs.csv")

print(labs.shape)       # 2,008 rows, 107 columns
display(labs.head())
labs.info()


(2008, 107)


,inpatient_number,body_temperature,pulse,respiration,systolic_blood_pressure,diastolic_blood_pressure,map_value,fio2,creatinine_enzymatic_method,urea,...,measured_residual_base,measured_bicarbonate,carboxyhemoglobin,body_temperature_blood_gas,oxygen_saturation,partial_oxygen_pressure,oxyhemoglobin,anion_gap,free_calcium,total_hemoglobin
0,857781,36.7,87,19,102,64,76.666667,33,108.3,12.55,...,-2.1,21.2,0.4,37.0,97.0,93.0,95.9,17.8,1.14,125.0
1,743087,36.8,95,18,150,70,96.666667,33,62.0,4.29,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,866418,36.5,98,18,102,67,78.666667,33,185.1,15.99,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,775928,36.0,73,19,110,74,86.000000,33,104.8,8.16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,810128,35.0,88,19,134,62,86.000000,33,83.9,6.86,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Columns: 107 entries, inpatient_number to total_hemoglobin
dtypes: float64(101), int64(6)
memory usage: 1.6 MB


### **2. Check missing values and duplicates**
**Reasoning :**  This file has no duplicate rows or patient IDs. Many lab columns have missing values because a test was not recorded for every patient. Do not replace all those blanks with zero.



In [7]:
missing = labs.isna().sum().sort_values(ascending=False)
print(missing)

print("Duplicate rows:", labs.duplicated().sum())
print("Duplicate patient IDs:", labs["inpatient_number"].duplicated().sum())


cholinesterase              2008
homocysteine                1862
apolipoprotein_a            1832
apolipoprotein_b            1832
lipoprotein                 1832
                            ... 
diastolic_blood_pressure       0
systolic_blood_pressure        0
respiration                    0
pulse                          0
inpatient_number               0
Length: 107, dtype: int64
Duplicate rows: 0
Duplicate patient IDs: 0


### **3. Remove the completely empty column**
**Reasoning:** The Cholinesterase column is completely empty and hence cannot be used for further analysis. So dropping the column from labs.



In [8]:
empty_columns = labs.columns[labs.isna().all()]
print(empty_columns.tolist())  # ['cholinesterase']

labs = labs.drop(columns=empty_columns)


['cholinesterase']


### **4. Mark unusable vital signs as missing**
**Reasoning:** Three rows have blood pressure recorded as zero; two have systolic pressure below diastolic pressure. Mark the blood pressure values and their calculated map_value as missing in those five rows.



In [9]:
invalid_bp = (
    (labs["systolic_blood_pressure"] == 0) |
    (labs["diastolic_blood_pressure"] == 0) |
    (labs["systolic_blood_pressure"] < labs["diastolic_blood_pressure"])
)

print("Rows with invalid blood pressure:", invalid_bp.sum())

labs.loc[
    invalid_bp,
    ["systolic_blood_pressure", "diastolic_blood_pressure", "map_value"]
] = np.nan

labs.loc[labs["pulse"] == 0, "pulse"] = np.nan
labs.loc[labs["respiration"] == 0, "respiration"] = np.nan


Rows with invalid blood pressure: 5
